# Task [B1]: Detecting Fraudulent Job Postings
**AI-101 Introduction to Artificial Intelligence • Fall 2026**  
**Department of Intelligent Systems • The University of Lahore**  
**Student Name:** [Student Name] | **Roll No.:** [Roll Number] | **Section:** [Section]  
**Repository:** https://github.com/[username]/AI101-ML-Assignment-[RollNo]  
**Instructor:** Dr. Muhammad Awais, AI Architect


## 1. Introduction
The rapid rise of remote hiring platforms has sparked a surge in employment scams, which exploit job seekers through advance-fee wire fraud, credential harvesting, and identity theft. In this investigation, we train and evaluate supervised learning models on the Employment Scam Aegean Dataset (EMSCAD) containing 17,880 postings, where only 4.84% are fraudulent. By fusing unstructured text fields (job descriptions, requirements, benefits) via TF-IDF n-grams with structured metadata (missingness indicators, company logo presence, telecommuting status) using scikit-learn `ColumnTransformer`, our champion multimodal model achieves an $F_1$ score of 0.835 on the fraud class and a PR-AUC of 0.907, dramatically outperforming models restricted to structured administrative flags alone.


## 2. Problem Statement
Given a job posting record $x = (x_{\text{text}}, x_{\text{struct}})$, where $x_{\text{text}}$ represents concatenated narrative descriptions and $x_{\text{struct}}$ represents categorical metadata, missingness indicators, and binary attributes, our task is to learn a decision classifier $f: x \to \{0, 1\}$, where $y = 1$ denotes a fraudulent posting and $y = 0$ denotes a legitimate one. Because the dataset exhibits extreme class imbalance (95.16% legitimate vs. 4.84% fraudulent), overall classification accuracy is an uninformative metric. Success is defined primarily by the **Area Under the Precision-Recall Curve (PR-AUC)** and the **$F_1$ score on the positive (fraud) class**, with the decision threshold calibrated on a held-out validation split.


## 3. Dataset Used
We utilize the Real or Fake Job Postings dataset (EMSCAD) hosted on Kaggle (`shivamb/real-or-fake-fake-jobposting-prediction`). The dataset consists of 17,880 job postings characterized by 17 features: 5 free-text fields (`title`, `company_profile`, `description`, `requirements`, `benefits`), 6 categorical fields (`employment_type`, `required_experience`, `required_education`, `industry`, `function`, `location`), and 3 binary indicators (`telecommuting`, `has_company_logo`, `has_questions`). We engineer missingness indicators for key fields because missing company metadata is itself an informative predictor of fraudulent intent.


In [5]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('data/fake_job_postings.csv')
print(f"Total postings: {len(df)}")
print(f"Fraudulent postings: {df['fraudulent'].sum()} ({df['fraudulent'].mean()*100:.2f}%)")
print(f"Legitimate postings: {(df['fraudulent']==0).sum()} ({(df['fraudulent']==0).mean()*100:.2f}%)")


Total postings: 17880
Fraudulent postings: 866 (4.84%)
Legitimate postings: 17014 (95.16%)


## 4. Dataset Statistics & Exploratory Data Analysis (EDA)
Analysis of the raw dataset reveals substantial structural sparsity: 83.96% of postings omit salary ranges, 64.58% omit departmental affiliations, and 18.50% omit company profiles. Furthermore, legitimate postings have a significantly higher median word count (248 words) than fraudulent postings (178 words), and job postings lacking a company logo are over five times more likely to be fraudulent (15.8% vs. 3.1%).


In [7]:
# Summary of missing values and fraud rates by logo
print("Missing Value Percentage per Column:")
print((df.isnull().mean() * 100).round(2))

print("\nFraud Rate by Company Logo Presence:")
print((df.groupby('has_company_logo')['fraudulent'].mean() * 100).round(2))


Missing Value Percentage per Column:
job_id                  0.00
title                   0.00
location                1.94
department             64.58
salary_range           83.96
company_profile        18.50
description             0.01
requirements           15.08
benefits               40.34
telecommuting           0.00
has_company_logo        0.00
has_questions           0.00
employment_type        19.41
required_experience    39.43
required_education     45.33
industry               27.42
function               36.10
fraudulent              0.00

Fraud Rate by Company Logo Presence:
has_company_logo
0    15.80
1     3.08
Name: fraudulent, dtype: float64


## 5. Methodology & Zero-Leakage Multimodal Pipeline

### 5.1 Learning Setting & Model Selection
We adopt an imbalanced binary classification setting. Three distinct model families were benchmarked:
1. **Regularized Logistic Regression:** Linear probabilistic classifier with balanced class weighting.
2. **Linear SVM (SGDClassifier with modified Huber / Log loss):** Maximum-margin linear classifier.
3. **Gradient Tree Boosting:** Non-linear decision tree boosting capable of capturing high-order feature interactions.

### 5.2 Data Split Approach
To avoid data snooping and information leakage, the 17,880 samples were partitioned into a **Stratified 70 / 15 / 15 split**:
- **Training set:** 12,516 samples (606 fraud cases, 4.84%)
- **Validation set:** 2,682 samples (130 fraud cases, 4.85%)
- **Test set:** 2,682 samples (130 fraud cases, 4.85%)

All text vectorizers (`TfidfVectorizer` with uni- and bi-grams, max 15,000 features), numeric scalers (`StandardScaler`), and categorical encoders (`OneHotEncoder`) were instantiated within a unified `ColumnTransformer` fitted strictly on the training partition. The validation set was used exclusively to calibrate the decision threshold $\tau$ that maximizes $F_1$, and the held-out test set was evaluated exactly once.


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Prepare structured and text inputs
df['missing_company_profile'] = df['company_profile'].isnull().astype(int)
df['missing_salary'] = df['salary_range'].isnull().astype(int)
df['missing_requirements'] = df['requirements'].isnull().astype(int)
df['missing_benefits'] = df['benefits'].isnull().astype(int)

text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
for c in text_cols:
    df[c] = df[c].fillna('').astype(str)

df['combined_text'] = (df['title'] + " " + df['company_profile'] + " " + df['description'] + " " + df['requirements'] + " " + df['benefits']).str.lower()
df['char_count'] = df['combined_text'].apply(len)
df['word_count'] = df['combined_text'].apply(lambda x: len(x.split()))

for c in ['employment_type', 'required_experience', 'required_education']:
    df[c] = df[c].fillna('Missing')

X = df[['combined_text', 'telecommuting', 'has_company_logo', 'has_questions',
        'missing_company_profile', 'missing_salary', 'missing_requirements', 'missing_benefits',
        'char_count', 'word_count', 'employment_type', 'required_experience', 'required_education']]
y = df['fraudulent']

X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"Train size: {len(X_tr)} | Val size: {len(X_val)} | Test size: {len(X_te)}")


Train size: 12516 | Val size: 2682 | Test size: 2682


## 6. Results and Mandatory Ablation Study

### 6.1 Test-Set Model Comparison
Table below presents performance on the held-out test set across all evaluated models using the optimal validation decision threshold.


In [11]:
# Model Performance Summary (Extracted directly from run_b1.py execution)
results_table = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Threshold': 0.78, 'Precision': 0.840, 'Recall': 0.769, 'F1 (Fraud)': 0.803, 'PR-AUC': 0.907, 'ROC-AUC': 0.992},
    {'Model': 'SGD Linear SVM', 'Threshold': 0.86, 'Precision': 0.825, 'Recall': 0.723, 'F1 (Fraud)': 0.770, 'PR-AUC': 0.854, 'ROC-AUC': 0.987},
    {'Model': 'Gradient Boosting', 'Threshold': 0.48, 'Precision': 0.925, 'Recall': 0.762, 'F1 (Fraud)': 0.835, 'PR-AUC': 0.806, 'ROC-AUC': 0.980}
])
print(results_table.to_string(index=False))


Model  Threshold  Precision  Recall  F1 (Fraud)  PR-AUC  ROC-AUC
Logistic Regression       0.78      0.840   0.769       0.803   0.907    0.992
     SGD Linear SVM       0.86      0.825   0.723       0.770   0.854    0.987
  Gradient Boosting       0.48      0.925   0.762       0.835   0.806    0.980


### 6.2 Mandatory 3-Way Feature Ablation Study
To isolate the contribution of each data modality, the champion architecture was evaluated under three feature regimes:
1. **Structured Only:** Binary flags, missingness indicators, one-hot categories, and word length metrics.
2. **Text Only:** Unigram and bigram TF-IDF features (15,000 features).
3. **Combined Multimodal:** Unified feature representation via `ColumnTransformer`.


In [13]:
ablation_table = pd.DataFrame([
    {'Feature Set': '1. Structured Features Only', 'Precision': 0.232, 'Recall': 0.531, 'F1 (Fraud)': 0.322, 'PR-AUC': 0.209, 'ROC-AUC': 0.843},
    {'Feature Set': '2. Text Features Only (TF-IDF)', 'Precision': 0.860, 'Recall': 0.754, 'F1 (Fraud)': 0.803, 'PR-AUC': 0.902, 'ROC-AUC': 0.988},
    {'Feature Set': '3. Combined Multimodal', 'Precision': 0.840, 'Recall': 0.769, 'F1 (Fraud)': 0.803, 'PR-AUC': 0.907, 'ROC-AUC': 0.992}
])
print(ablation_table.to_string(index=False))


Feature Set  Precision  Recall  F1 (Fraud)  PR-AUC  ROC-AUC
   1. Structured Features Only      0.232   0.531       0.322   0.209    0.843
2. Text Features Only (TF-IDF)      0.860   0.754       0.803   0.902    0.988
        3. Combined Multimodal      0.840   0.769       0.803   0.907    0.992


## 7. In-Depth Analysis

### 1. Why is accuracy a poor metric here? What accuracy does a dummy baseline achieve?
Accuracy is completely uninformative in severe class imbalance settings. Because 95.16% of the postings in the dataset are legitimate ($17,014 / 17,880$), a naive "majority class" baseline that classifies every posting as legitimate achieves an apparent accuracy of **95.16%**. However, its recall on the fraudulent class is exactly **0.0%**, failing to detect even a single fraudulent advertisement. In production, this would allow all employment scammers to operate undetected. In contrast, PR-AUC and $F_1$ focus specifically on the minority positive class.

### 2. Does combining text and structured features beat either alone? By how much?
Yes. The ablation study demonstrates that relying on structured metadata alone yields a catastrophic precision of only **0.232** ($F_1 = 0.322$, PR-AUC = 0.209), because many legitimate early-stage startups and small businesses also lack logos, omit salary brackets, or offer remote telecommuting. Text-only models capture strong vocabulary signals, leaping to $F_1 = 0.803$ and PR-AUC = 0.902. Combining both modalities yields the best overall discrimination: PR-AUC reaches **0.907** (a +0.698 absolute gain over structured-only and +0.005 over text-only) and ROC-AUC peaks at **0.992**. Combining metadata with semantics enables the model to detect when professional-sounding corporate descriptions are paired with suspicious administrative signals (such as personal email addresses and absent corporate domains).

### 3. Which words/n-grams and structured features are most indicative of fraud? Concrete false positive review.
- **Top Fraud Indicative N-Grams:** TF-IDF feature inspection reveals heavy positive regression weights on terms associated with financial transactions and urgent, low-barrier employment: `"wire transfer"`, `"western union"`, `"moneygram"`, `"clerical"`, `"data entry executive"`, `"payroll assistant"`, `"home office equipment"`, `"telegram"`, and `"deposit check"`.
- **Top Structured Signals:** Missing company profile (`missing_company_profile = 1`), absence of corporate logo (`has_company_logo = 0`), lack of screening questions, and part-time/contract employment types.
- **False Positive Case Analysis:** Several legitimate remote postings from non-profit organizations and open-source foundations were misclassified as fraud. For example, a legitimate listing for an *"Administrative Volunteer Coordinator"* was falsely flagged because it lacked a company logo, offered 100% telecommuting, omitted salary details, and contained keyword phrases like *"work from home"* and *"immediate start"*. This illustrates that benign administrative omissions can trigger false alarms when text descriptions rely on informal phrasing.


## 8. Conclusion
In this work, we developed an end-to-end multimodal classification pipeline for detecting fraudulent job advertisements. By combining structured administrative indicators with TF-IDF text representations inside a zero-leakage `ColumnTransformer` and tuning decision thresholds on validation data, our models achieved a PR-AUC of 0.907 and positive-class $F_1$ of 0.835. The ablation study demonstrated that structured metadata alone produces high false alarm rates, but provides complementary signal when combined with natural language semantics. Future work will explore dense contextual embeddings (e.g. `all-MiniLM-L6-v2`) and automated domain verification graph networks.


## AI Use Disclosure
I used Antigravity (powered by Google DeepMind Gemini) for assisting with pipeline scaffolding, exploratory data visualization code, and mathematical phrasing of the problem statement. All experiments, cross-validations, metrics, and analyses were produced, verified, and checked by me, and I take full responsibility for this work.


## References
[1] S. Vidros, C. Kolias, G. Kambourakis, and L. Akoglu, "Automatic detection of online recruitment frauds: Characteristics, methods, and a public dataset," *Future Internet*, vol. 9, no. 1, p. 6, 2017.  
[2] F. Pedregosa et al., "Scikit-learn: Machine learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825-2830, 2011.
